# https://www.minecraft-schematics.com

## Получение ссылок

In [ ]:
import requests
import json
import time
from tqdm import tqdm
from bs4 import BeautifulSoup


def fetch_page_links(page_url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    try:
        resp = requests.get(page_url, headers=headers, timeout=15)
        resp.raise_for_status()
    except Exception as e:
        print(f"    Ошибка загрузки: {e}")
        return []
    soup = BeautifulSoup(resp.text, 'html.parser')
    links = []
    for block in soup.find_all('div', class_='span4'):
        has_schematic = block.find('button', class_='format_schematic')
        has_non_free = block.find('button', class_='format_non_free')
        if has_schematic and not has_non_free:
            a = block.find('h3').find('a') if block.find('h3') else None
            if a and a.has_attr('href'):
                href = a['href']
                if not href.startswith('http'):
                    href = 'https://www.minecraft-schematics.com' + href
                links.append(href)
    return links


def main():
    total_pages = 1141
    base_list_url = 'https://www.minecraft-schematics.com/latest/'
    all_links = []
    output_file = 'all_free_schematics.json'

    pbar = tqdm(total=total_pages, desc='Сбор ссылок', unit='стр')
    for page in range(1, total_pages + 1):
        if page == 1:
            url = base_list_url
        else:
            url = f"{base_list_url}{page}/"

        page_links = fetch_page_links(url)
        all_links.extend(page_links)

        pbar.set_description(f'Стр.{page}: +{len(page_links)} ссылок (всего {len(all_links)})')
        pbar.update(1)

        # Промежуточное сохранение
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(all_links, f, indent=4, ensure_ascii=False)

        time.sleep(0.25)  # пауза для бережного отношения к серверу

    print(f"\n✅ Сбор завершён! Всего собрано {len(all_links)} ссылок.")
    print("Примеры ссылок (первые 5):")
    for link in all_links[:5]:
        print(link)

main()

Стр.1: +0 ссылок (всего 0):   0%|          | 1/1141 [00:00<04:03,  4.68стр/s]

    Ошибка загрузки: 403 Client Error: Forbidden for url: https://www.minecraft-schematics.com/latest/


Стр.2: +0 ссылок (всего 0):   0%|          | 2/1141 [00:00<07:41,  2.47стр/s]

    Ошибка загрузки: 403 Client Error: Forbidden for url: https://www.minecraft-schematics.com/latest/2/


Стр.3: +0 ссылок (всего 0):   0%|          | 3/1141 [00:01<07:51,  2.41стр/s]

...

Стр.5: +0 ссылок (всего 0):   0%|          | 5/1141 [00:02<08:11,  2.31стр/s]

    Ошибка загрузки: 403 Client Error: Forbidden for url: https://www.minecraft-schematics.com/latest/5/


Стр.6: +0 ссылок (всего 0):   1%|          | 6/1141 [00:02<08:10,  2.31стр/s]

    Ошибка загрузки: 403 Client Error: Forbidden for url: https://www.minecraft-schematics.com/latest/6/


KeyboardInterrupt: 

In [11]:
import os
from pathlib import Path
import json

links = []

for file in os.listdir("../../data/web_data"):
    file = Path(file)
    links.append(f"https://www.minecraft-schematics.com/schematic/{file.stem}/")


with open("all_free_schematics.json", "w") as f:
    json.dump(links, f, indent=4)

## Скачивание файлов

In [ ]:
import os
import time
import re
import json
import subprocess
import threading
import shutil
import undetected_chromedriver as uc
from tqdm import tqdm


def get_chrome_version():
    """Определяет major-версию Chrome."""
    paths = [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        os.path.expanduser(r"~\AppData\Local\Google\Chrome\Application\chrome.exe"),
    ]
    for path in paths:
        if os.path.exists(path):
            try:
                cmd = f'powershell -command "(Get-Item \'{path}\').VersionInfo.FileVersion"'
                result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
                version = result.stdout.strip()
                if version:
                    return int(version.split(".")[0])
            except Exception:
                continue
    try:
        cmd = r'reg query "HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon" /v version'
        result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
        match = re.search(r"(\d+)\.\d+\.\d+\.\d+", result.stdout)
        if match:
            return int(match.group(1))
    except Exception:
        pass
    return int(input("Введи major-версию Chrome (например 145): ").strip())


# def create_driver(save_dir: str, chrome_version: int):
#     """Создаёт Chrome с нужной версией ChromeDriver."""
#     options = uc.ChromeOptions()
#     prefs = {
#         "download.default_directory": save_dir,
#         "download.prompt_for_download": False,
#         "download.directory_upgrade": True,
#         "safebrowsing.enabled": True,
#     }
#     options.add_experimental_option("prefs", prefs)
#     return uc.Chrome(options=options, version_main=chrome_version)


def create_driver(save_dir: str, chrome_version: int, worker_id: int = 0):
    options = uc.ChromeOptions()

    prefs = {
        "download.default_directory": save_dir,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    # Отдельный временный профиль для каждого воркера
    temp_profile = os.path.abspath(os.path.join(os.getcwd(), f"_chrome_profile_{worker_id}"))

    return uc.Chrome(
        options=options,
        version_main=chrome_version,
        user_data_dir=temp_profile,  # ← изолированный профиль
    )


def wait_for_download(save_dir: str, files_before: set, timeout: int = 120):
    """Ждёт появления нового файла в папке."""
    elapsed = 0
    while elapsed < timeout:
        time.sleep(0.2)
        elapsed += 0.2
        files_now = set(os.listdir(save_dir))
        new_files = files_now - files_before
        completed = [f for f in new_files
                     if not f.endswith((".crdownload", ".tmp", ".part"))]
        if completed:
            time.sleep(0.2)
            return completed[0]
    return None


def download_single(driver, schematic_url: str, save_dir: str, timeout: int = 120) -> bool:
    """Скачивает одну схематику."""
    try:
        if not schematic_url.endswith("/"):
            schematic_url += "/"

        # driver.get(schematic_url)
        # time.sleep(1)

        download_url = schematic_url + "download/action/?type=schematic"
        files_before = set(os.listdir(save_dir))
        driver.get(download_url)

        downloaded_file = wait_for_download(save_dir, files_before, timeout)

        if downloaded_file:
            filepath = os.path.join(save_dir, downloaded_file)
            file_size = os.path.getsize(filepath)

            # Проверяем что скачался не HTML
            if file_size < 5000:
                try:
                    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                        start = f.read(200).strip().lower()
                        if start.startswith("<!doctype") or start.startswith("<html"):
                            os.remove(filepath)
                            return False
                except Exception:
                    pass
            return True
        return False
    except Exception:
        return False


def save_failed(failed_list: list, filepath: str = "failed.json"):
    """Сохраняет неудачные ссылки."""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(failed_list, f, indent=2, ensure_ascii=False)


def split_urls(urls: list, n: int) -> list:
    """Делит список на n примерно равных частей."""
    k, m = divmod(len(urls), n)
    return [urls[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


def worker_func(driver, urls_chunk, save_dir, failed, success_count, lock, pbar):
    """Функция одного воркера — скачивает свою часть ссылок."""
    for url in urls_chunk:
        ok = download_single(driver, url, save_dir)

        with lock:
            pbar.update(1)
            if ok:
                success_count[0] += 1
            else:
                failed.append(url)
                save_failed(failed)
            pbar.set_description(f"✅ {success_count[0]} ❌ {len(failed)}")

        time.sleep(0.5)


def main(num_workers: int = None):
    # input_file = "all_free_schematics.json"
    input_file="failed_copy.json"
    if not os.path.exists(input_file):
        print(f"❌ Файл {input_file} не найден!")
        return

    with open(input_file, "r", encoding="utf-8") as f:
        urls = json.load(f)

    total = len(urls)

    if num_workers is None:
        num_workers = int(input(f"Количество воркеров (всего {total} ссылок): ").strip())
    num_workers = min(num_workers, total)

    # Общая папка и подпапки для каждого воркера
    base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../data/web_data/"))
    os.makedirs(base_dir, exist_ok=True)

    worker_dirs = []
    for i in range(num_workers):
        d = os.path.abspath(os.path.join(base_dir, f"_worker_{i}"))
        os.makedirs(d, exist_ok=True)
        worker_dirs.append(d)

    # Определяем версию Chrome один раз
    chrome_version = get_chrome_version()

    # Запускаем все браузеры и открываем страницу логина
    drivers = []
    for i in range(num_workers):
        print(f"🚀 Запускаю Chrome #{i + 1}/{num_workers}...")
        driver = create_driver(worker_dirs[i], chrome_version)
        driver.get("https://www.minecraft-schematics.com/login/")
        drivers.append(driver)
        time.sleep(2)

    input(f"\n🔑 Залогинься во всех {num_workers} браузерах и нажми Enter... ")

    # Делим ссылки между воркерами
    chunks = split_urls(urls, num_workers)

    # Общее состояние
    failed = []
    success_count = [0]  # list чтобы можно было менять из потока
    lock = threading.Lock()
    pbar = tqdm(total=total, desc="✅ 0 ❌ 0", unit="файл")

    # Запускаем потоки
    threads = []
    for i in range(num_workers):
        t = threading.Thread(
            target=worker_func,
            args=(drivers[i], chunks[i], worker_dirs[i],
                  failed, success_count, lock, pbar),
            daemon=True,
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    pbar.close()

    # Переносим все файлы в общую папку downloads/
    for d in worker_dirs:
        for filename in os.listdir(d):
            src = os.path.join(d, filename)
            dst = os.path.join(base_dir, filename)
            # Если файл с таким именем уже есть — добавляем суффикс
            if os.path.exists(dst):
                name, ext = os.path.splitext(filename)
                counter = 1
                while os.path.exists(dst):
                    dst = os.path.join(base_dir, f"{name}_{counter}{ext}")
                    counter += 1
            shutil.move(src, dst)
        # Удаляем пустую папку воркера
        try:
            os.rmdir(d)
        except OSError:
            pass

    # Итоги
    print(f"\n✅ Успешно: {success_count[0]}/{total}")
    print(f"❌ Неудачно: {len(failed)}/{total}")
    print(f"📁 Папка: {base_dir}")

    if failed:
        save_failed(failed)
        print("💾 Неудачные ссылки: failed.json")
    elif os.path.exists("failed.json"):
        os.remove("failed.json")

    # Закрываем браузеры
    for driver in drivers:
        try:
            driver.quit()
        except Exception:
            pass


if __name__ == "__main__":
    main(1)

🚀 Запускаю Chrome #1/1...


✅ 0 ❌ 10: 100%|██████████| 10/10 [21:26<00:00, 128.70s/файл]



✅ Успешно: 0/10
❌ Неудачно: 10/10
📁 Папка: c:\Users\ryabo\Documents\Диплом\DiffusionCraft\data\web_data
💾 Неудачные ссылки: failed.json


: 

### Скачивание описания

In [1]:
import os
import time
import re
import json
import subprocess
import threading
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
from tqdm import tqdm


def get_chrome_version():
    paths = [
        r"C:\Program Files\Google\Chrome\Application\chrome.exe",
        r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
        os.path.expanduser(r"~\AppData\Local\Google\Chrome\Application\chrome.exe"),
    ]
    for path in paths:
        if os.path.exists(path):
            try:
                cmd = f'powershell -command "(Get-Item \'{path}\').VersionInfo.FileVersion"'
                result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
                version = result.stdout.strip()
                if version:
                    return int(version.split(".")[0])
            except Exception:
                continue
    try:
        cmd = r'reg query "HKEY_CURRENT_USER\Software\Google\Chrome\BLBeacon" /v version'
        result = subprocess.run(cmd, capture_output=True, text=True, shell=True)
        match = re.search(r"(\d+)\.\d+\.\d+\.\d+", result.stdout)
        if match:
            return int(match.group(1))
    except Exception:
        pass
    return int(input("Введи major-версию Chrome (например 145): ").strip())


def create_driver(chrome_version: int):
    options = uc.ChromeOptions()
    return uc.Chrome(options=options, version_main=chrome_version)


def is_error_page(html: str) -> str | None:
    """
    Проверяет, является ли страница ошибкой.
    Возвращает тип ошибки или None если страница нормальная.
    """
    lower = html.lower()

    # Cloudflare rate limit
    if "error 1015" in lower or "rate limit" in lower or "you are being rate limited" in lower:
        return "rate_limit"

    # Другие ошибки Cloudflare
    if "error 1" in lower and "cloudflare" in lower:
        return "cloudflare"

    # Access denied
    if "error 403" in lower or "access denied" in lower or "error 401" in lower:
        return "access_denied"

    # Проверяем title на ошибки
    soup = BeautifulSoup(html, "html.parser")
    title = soup.find("title")
    if title:
        title_text = title.get_text(strip=True).lower()
        if re.match(r"error\s*\d+", title_text):
            return "error"
        if "just a moment" in title_text:
            return "cloudflare_challenge"

    return None


def parse_schematic_page(html: str) -> dict:
    """Извлекает название и описание из HTML."""
    soup = BeautifulSoup(html, "html.parser")

    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else None

    description = None
    for legend in soup.find_all("legend"):
        if "description" in legend.get_text(strip=True).lower():
            parent = legend.find_parent("div")
            if parent:
                desc_parts = []
                for p in parent.find_all("p"):
                    if p.find("legend"):
                        continue
                    for br in p.find_all("br"):
                        br.replace_with("\n")
                    text = p.get_text(strip=False).strip()
                    if text:
                        desc_parts.append(text)
                description = "\n".join(desc_parts) if desc_parts else None
            break

    return {"title": title, "description": description}


def scrape_with_retry(driver, url: str, max_retries: int = 5) -> dict | None:
    """
    Открывает страницу, проверяет на ошибки, при rate limit ждёт и повторяет.
    """
    for attempt in range(max_retries):
        try:
            driver.get(url)
            time.sleep(2)

            html = driver.page_source
            error = is_error_page(html)

            if error == "rate_limit":
                # Экспоненциальный откат: 30, 60, 120, 240, 480 сек
                wait_time = 30 * (2 ** attempt)
                time.sleep(wait_time)
                continue

            if error == "cloudflare_challenge":
                # Ждём пока Cloudflare пропустит
                time.sleep(10)
                html = driver.page_source
                error = is_error_page(html)
                if error:
                    time.sleep(20)
                    continue

            if error:
                # Другие ошибки — тоже ждём и пробуем снова
                wait_time = 15 * (2 ** attempt)
                time.sleep(wait_time)
                continue

            # Страница нормальная — парсим
            data = parse_schematic_page(html)
            data["url"] = url

            # Дополнительная проверка: title должен быть осмысленным
            if not data["title"]:
                time.sleep(5)
                continue

            return data

        except Exception:
            time.sleep(10)
            continue

    return None


def split_urls(urls: list, n: int) -> list:
    k, m = divmod(len(urls), n)
    return [urls[i * k + min(i, m):(i + 1) * k + min(i + 1, m)] for i in range(n)]


def worker_func(driver, urls_chunk, results, failed, lock, pbar):
    """Один воркер — парсит свою часть ссылок."""
    for url in urls_chunk:
        data = scrape_with_retry(driver, url)

        with lock:
            if data and data["title"]:
                results.append(data)
            else:
                failed.append(url)

            pbar.update(1)
            pbar.set_description(f"✅ {len(results)} ❌ {len(failed)}")

            # Промежуточное сохранение
            if len(results) % 50 == 0 and results:
                with open("schematics_info.json", "w", encoding="utf-8") as f:
                    json.dump(results, f, indent=2, ensure_ascii=False)
            if failed:
                with open("failed_scrape.json", "w", encoding="utf-8") as f:
                    json.dump(failed, f, indent=2, ensure_ascii=False)

        time.sleep(0.5)


def main(num_workers: int = None):
    input_file = "all_free_schematics.json"

    if not os.path.exists(input_file):
        print(f"❌ Файл {input_file} не найден!")
        return

    with open(input_file, "r", encoding="utf-8") as f:
        urls = json.load(f)

    total = len(urls)

    if num_workers is None:
        num_workers = int(input(f"Количество воркеров (всего {total} ссылок): ").strip())
    num_workers = min(num_workers, total)

    chrome_version = get_chrome_version()

    drivers = []
    for i in range(num_workers):
        print(f"🚀 Запускаю Chrome #{i + 1}/{num_workers}...")
        driver = create_driver(chrome_version)
        driver.get("https://www.minecraft-schematics.com/")
        drivers.append(driver)
        time.sleep(2)

    input(f"\n🔑 Пройди Cloudflare во всех {num_workers} браузерах и нажми Enter... ")

    chunks = split_urls(urls, num_workers)

    results = []
    failed = []
    lock = threading.Lock()
    pbar = tqdm(total=total, desc="✅ 0 ❌ 0", unit="стр")

    threads = []
    for i in range(num_workers):
        t = threading.Thread(
            target=worker_func,
            args=(drivers[i], chunks[i], results, failed, lock, pbar),
            daemon=True,
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    pbar.close()

    with open("schematics_info.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    if failed:
        with open("failed_scrape.json", "w", encoding="utf-8") as f:
            json.dump(failed, f, indent=2, ensure_ascii=False)

    print(f"\n✅ Успешно: {len(results)}/{total}")
    print(f"❌ Неудачно: {len(failed)}/{total}")
    print(f"💾 Результат: schematics_info.json")
    if failed:
        print(f"💾 Неудачные: failed_scrape.json")

    for driver in drivers:
        try:
            driver.quit()
        except Exception:
            pass

    print("🔒 Все браузеры закрыты.")


if __name__ == "__main__":
    main()

🚀 Запускаю Chrome #1/4...
🚀 Запускаю Chrome #2/4...
🚀 Запускаю Chrome #3/4...
🚀 Запускаю Chrome #4/4...


✅ 14576 ❌ 0: 100%|██████████| 14576/14576 [4:04:25<00:00,  1.01s/стр]  



✅ Успешно: 14576/14576
❌ Неудачно: 0/14576
💾 Результат: schematics_info.json
🔒 Все браузеры закрыты.


# Веб-сайт https://abfielder.com/Products/BrowseProducts.php

In [4]:
import requests
from bs4 import BeautifulSoup
import time
import json
from urllib.parse import urljoin
from tqdm import tqdm  # 需要安装：pip install tqdm

BASE_URL = "https://abfielder.com/Products/BrowseProducts.php"
PRODUCT_PAGE_BASE = "https://abfielder.com/Products/"

# Параметры запроса
PARAMS = {
    "sort": "trending",
    "game": "minecraft",
    "productType": "1"  # 1 = Schematics (бесплатные)
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_soup(url, params=None):
    """Выполняет GET-запрос и возвращает объект BeautifulSoup."""
    try:
        response = requests.get(url, params=params, headers=HEADERS, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.RequestException as e:
        print(f"\nОшибка при запросе {url}: {e}")
        return None

def extract_product_links(soup):
    """
    Извлекает ссылки на бесплатные постройки со страницы.
    Пропускает платные товары (с классом bg-danger-subtle).
    """
    links = []
    if not soup:
        return links

    # Ищем все ссылки, содержащие 'ProductDetails.php'
    product_links = soup.select("a[href*='ProductDetails.php']")
    
    for a_tag in product_links:
        # Проверяем, не обёрнута ли карточка в платный блок.
        card_div = a_tag.find("div", class_="card")
        if card_div and "bg-danger-subtle" in card_div.get("class", []):
            continue  # Пропускаем платные товары
        
        href = a_tag.get("href")
        # Исключаем редиректы на премиум (на всякий случай)
        if href and "redirectPremium" not in href:
            full_url = urljoin(PRODUCT_PAGE_BASE, href)
            links.append(full_url)
    
    return links

def get_last_page_from_pagination(soup):
    """Извлекает номер последней страницы из пагинации."""
    if not soup:
        return None
    # Ищем ссылки в пагинации
    page_links = soup.select("ul.pagination li.page-item a.page-link")
    max_page = 1
    for link in page_links:
        text = link.get_text(strip=True)
        if text.isdigit():
            page_num = int(text)
            if page_num > max_page:
                max_page = page_num
    return max_page if max_page > 1 else None

def main():
    all_links = []
    max_pages = 173  # начальное предположение

    # Определяем реальное количество страниц
    print("Определение количества страниц...")
    first_page_soup = get_soup(BASE_URL, {**PARAMS, "page": 1})
    if first_page_soup:
        detected_pages = get_last_page_from_pagination(first_page_soup)
        if detected_pages:
            max_pages = detected_pages
            print(f"Обнаружено страниц: {max_pages}")
        else:
            print(f"Не удалось определить число страниц, используем {max_pages}")
    else:
        print("Не удалось загрузить первую страницу, выход.")
        return

    # Используем tqdm для отображения прогресса по страницам
    for page in tqdm(range(1, max_pages + 1), desc="Обработка страниц", unit="стр"):
        params = PARAMS.copy()
        params["page"] = page
        
        soup = get_soup(BASE_URL, params)
        if not soup:
            continue  # просто пропускаем проблемную страницу
        
        links = extract_product_links(soup)
        all_links.extend(links)
        
        # Можно обновлять описание прогресс-бара текущей информацией
        # tqdm.write(f"Страница {page}: найдено ссылок {len(links)} (всего {len(all_links)})")
        
        time.sleep(0.5)  # Вежливая пауза

    # Удаляем дубликаты
    unique_links = list(dict.fromkeys(all_links))
    print(f"\nВсего собрано уникальных ссылок: {len(unique_links)}")

    # Сохраняем в JSON
    with open("minecraft_builds_links.json", "w", encoding="utf-8") as f:
        json.dump(unique_links, f, indent=2, ensure_ascii=False)

    print("Ссылки сохранены в файл minecraft_builds_links.json")

if __name__ == "__main__":
    main()

Определение количества страниц...
Обнаружено страниц: 173


Обработка страниц: 100%|██████████| 173/173 [03:31<00:00,  1.22s/стр]


Всего собрано уникальных ссылок: 4655
Ссылки сохранены в файл minecraft_builds_links.json


In [9]:
import json
import time
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# Приоритет расширений (чем меньше число, тем выше приоритет)
EXT_PRIORITY = {
    '.schem': 1,
    '.litematic': 2,
    '.schematic': 3,
    '.zip': 4
}

def get_soup(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        return BeautifulSoup(resp.text, 'html.parser')
    except Exception as e:
        print(f"Ошибка загрузки {url}: {e}")
        return None

def extract_title(soup):
    title_tag = soup.select_one('h1.h4.fw-bold.mb-1')
    if title_tag:
        return title_tag.get_text(strip=True)
    title_tag = soup.find('h1')
    return title_tag.get_text(strip=True) if title_tag else None

def extract_description(soup):
    card_body = soup.select_one('.card-body')
    if not card_body:
        return ""
    description_parts = []
    for elem in card_body.children:
        if elem.name == 'p':
            if elem.find('a', class_='badge'):
                break
            text = elem.get_text(strip=True)
            if text:
                description_parts.append(text)
        elif elem.name == 'div' and ('Tags' in elem.get_text() or 'Dimensions' in elem.get_text()):
            break
        elif elem.name == 'h6' and ('Tags' in elem.get_text() or 'Dimensions' in elem.get_text()):
            break
    return '\n'.join(description_parts).strip()

def parse_version(ver_str):
    numbers = re.findall(r'\d+', ver_str)
    return tuple(int(n) for n in numbers) if numbers else (0,)

def extract_files_info(soup):
    """Возвращает словарь версий, где каждая версия содержит список файлов с полями filename, ext, url."""
    table = soup.select_one('table.table')
    if not table:
        return {}
    versions = {}
    rows = table.select('tbody tr')
    for row in rows:
        tds = row.find_all('td')
        if len(tds) < 3:
            continue
        first_td = tds[0]
        filename_text = ''
        for content in first_td.contents:
            if isinstance(content, str):
                filename_text += content.strip()
            elif content.name == 'br':
                break
        filename = filename_text.strip()
        if not filename:
            continue
        ext = None
        for e in EXT_PRIORITY:
            if filename.lower().endswith(e):
                ext = e
                break
        if not ext:
            continue
        version = tds[1].get_text(strip=True)
        download_btn = tds[2].find('a', class_='btn-primary', string=lambda s: s and 'Download' in s)
        if not download_btn:
            continue
        href = download_btn.get('href')
        if not href:
            continue
        full_url = urljoin("https://abfielder.com/Products/", href)
        if version not in versions:
            versions[version] = []
        versions[version].append({
            'filename': filename,
            'ext': ext,
            'url': full_url
        })
    return versions

def select_best_file(versions):
    """
    Выбирает файл из последней версии с наивысшим приоритетом расширения.
    Возвращает (filename, download_url, ext) или (None, None, None).
    """
    if not versions:
        return None, None, None
    # Сортируем версии по убыванию номера
    sorted_versions = sorted(versions.keys(), key=lambda v: parse_version(v), reverse=True)
    latest = sorted_versions[0]
    files = versions[latest]
    # Сортируем файлы по приоритету расширения
    files_sorted = sorted(files, key=lambda f: EXT_PRIORITY[f['ext']])
    best = files_sorted[0]
    return best['filename'], best['url'], best['ext']

def process_product(url):
    soup = get_soup(url)
    if not soup:
        return None
    title = extract_title(soup)
    description = extract_description(soup)
    versions = extract_files_info(soup)
    filename, download_url, ext = select_best_file(versions)
    if not download_url:
        print(f"Не найдена ссылка для скачивания на {url}")
        return None
    return {
        'product_url': url,
        'title': title,
        'description': description,
        'download_url': download_url,
        'filename': filename,
        'ext': ext
    }

def main():
    try:
        with open('minecraft_builds_links.json', 'r', encoding='utf-8') as f:
            links = json.load(f)
    except FileNotFoundError:
        print("Файл minecraft_builds_links.json не найден. Сначала запустите сбор ссылок.")
        return

    print(f"Загружено {len(links)} ссылок на продукты.")

    schematics_results = []
    zip_results = []

    for link in tqdm(links, desc="Обработка продуктов"):
        data = process_product(link)
        if not data:
            continue
        if data['ext'] == '.zip':
            zip_results.append({
                'product_url': data['product_url'],
                'title': data['title'],
                'description': data['description'],
                'download_url': data['download_url'],
                'filename': data['filename']
            })
        else:
            schematics_results.append({
                'product_url': data['product_url'],
                'title': data['title'],
                'description': data['description'],
                'download_url': data['download_url'],
                'filename': data['filename']
            })
        time.sleep(0.25)

    print(f"Успешно обработано: схем - {len(schematics_results)}, ZIP - {len(zip_results)}")

    with open('minecraft_schematics_data.json', 'w', encoding='utf-8') as f:
        json.dump(schematics_results, f, indent=2, ensure_ascii=False)

    with open('minecraft_zips_data.json', 'w', encoding='utf-8') as f:
        json.dump(zip_results, f, indent=2, ensure_ascii=False)

    print("Данные сохранены в minecraft_schematics_data.json и minecraft_zips_data.json")

if __name__ == "__main__":
    main()

Загружено 4655 ссылок на продукты.


Обработка продуктов:   0%|          | 0/4655 [00:00<?, ?it/s]

Обработка продуктов:   5%|▍         | 220/4655 [03:01<1:03:26,  1.16it/s]

Не найдена ссылка для скачивания на https://abfielder.com/Products/ProductDetails.php?id=11489


Обработка продуктов:  13%|█▎        | 588/4655 [07:58<51:14,  1.32it/s]  

...

Не найдена ссылка для скачивания на https://abfielder.com/Products/ProductDetails.php?id=2921


Обработка продуктов:  74%|███████▍  | 3448/4655 [46:11<14:23,  1.40it/s]

Не найдена ссылка для скачивания на https://abfielder.com/Products/ProductDetails.php?id=2884


Обработка продуктов: 100%|██████████| 4655/4655 [1:01:53<00:00,  1.25it/s]

Не найдена ссылка для скачивания на https://abfielder.com/Products/ProductDetails.php?id=11394
Успешно обработано: схем - 4538, ZIP - 103
Данные сохранены в minecraft_schematics_data.json и minecraft_zips_data.json


In [1]:
import json

with open("minecraft_zips_data.json") as f:
    print(len(json.load(f)))

103


In [1]:
!pip install DrissionPage

    scikit-image (<0.19.0>=0.16.1) ; extra == 'all'
                 ~~~~~~~~^

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import time
from pathlib import Path
from datetime import datetime
from DrissionPage import ChromiumPage, ChromiumOptions
from tqdm import tqdm

# === НАСТРОЙКИ ===
BASE_DOWNLOAD_DIR = Path("./downloads").absolute()
BASE_DOWNLOAD_DIR.mkdir(exist_ok=True)

LOG_DIR = Path("./logs").absolute()
LOG_DIR.mkdir(exist_ok=True)

AUTO_DOWNLOAD_WAIT = 30       # секунд ожидания начала загрузки (уменьшено)
DOWNLOAD_TIMEOUT = 20       # макс. время ожидания после клика по fallback
RETRIES = 1                   # повторных попыток при неудаче
SCHEMA_EXTENSIONS = {'.litematic', '.schem', '.schematic', '.nbt'}

def log_message(msg, log_file=None):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_msg = f"[{timestamp}] {msg}"
    tqdm.write(full_msg)
    if log_file:
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(full_msg + "\n")

def sanitize_folder_name(name):
    safe = "".join(c for c in name if c.isalnum() or c in "._- ()").strip()
    return safe if safe else "unknown"

def get_current_files(directory):
    return {f.name for f in directory.iterdir() if f.is_file()}

def wait_for_any_new_file(directory, before_files, timeout):
    start = time.time()
    while time.time() - start < timeout:
        time.sleep(0.2)  # чаще проверяем
        current = get_current_files(directory)
        new = current - before_files
        if new:
            return list(new)[0]
    return None

def create_page(download_dir):
    co = ChromiumOptions()
    co.set_pref("download.default_directory", str(download_dir))
    co.set_pref("download.prompt_for_download", False)
    co.set_pref("download.directory_upgrade", True)
    co.set_argument("--disable-blink-features=AutomationControlled")
    co.set_argument("--no-sandbox")
    return ChromiumPage(co)

def set_download_directory(page, download_dir):
    try:
        page.run_cdp("Page.setDownloadBehavior", behavior="allow", downloadPath=str(download_dir))
    except Exception as e:
        log_message(f"  Не удалось изменить папку загрузки через CDP: {e}", None)

def click_fallback_download(page, log_file):
    try:
        ele = page.wait.ele_displayed('#fallbackLink', timeout=2)
        if not ele:
            ele = page.wait.ele_displayed('@text():click here to download', timeout=2)
        if not ele:
            ele = page.wait.ele_displayed('@href*=PartnerUploadedFiles', timeout=2)
        if ele:
            page.scroll.to_see(ele)
            time.sleep(0.2)
            ele.click()
            log_message("  Нажата fallback-ссылка", log_file)
            return True
    except Exception as e:
        log_message(f"  Fallback-ссылка не найдена: {e}", log_file)
    return False

def initiate_download(page, url, download_dir, log_file):
    set_download_directory(page, download_dir)
    files_before = get_current_files(download_dir)

    page.get(url)
    log_message(f"  Переход по ссылке...", log_file)
    page.wait.doc_loaded()  # ждём только DOM, не все ресурсы

    log_message(f"  Ожидание начала загрузки (до {AUTO_DOWNLOAD_WAIT} сек)...", log_file)
    new_file = wait_for_any_new_file(download_dir, files_before, timeout=AUTO_DOWNLOAD_WAIT)
    if new_file:
        log_message(f"  Загрузка началась: {new_file}", log_file)
        return True

    log_message("  Автоматическая загрузка не началась, ищем fallback-ссылку...", log_file)
    if click_fallback_download(page, log_file):
        new_file = wait_for_any_new_file(download_dir, files_before, timeout=DOWNLOAD_TIMEOUT)
        if new_file:
            log_message(f"  Загрузка началась после fallback: {new_file}", log_file)
            return True
        else:
            log_message("  Загрузка не началась и после fallback", log_file)
            return False
    else:
        log_message("  Fallback-ссылка не найдена, загрузка не началась", log_file)
        return False

def check_folder_completed(folder_path: Path) -> bool:
    files = list(folder_path.iterdir())
    if not files:
        return False
    has_schema = any(f.suffix.lower() in SCHEMA_EXTENSIONS for f in files if f.is_file())
    has_crdownload = any(f.suffix.lower() == '.crdownload' for f in files)
    return has_schema and not has_crdownload

def finalize_results(data):
    successful = []
    failed = []
    for item in data:
        filename = item.get('filename', 'unknown')
        folder_name = sanitize_folder_name(filename)
        folder_path = BASE_DOWNLOAD_DIR / folder_name
        if folder_path.exists() and check_folder_completed(folder_path):
            successful.append(item)
        else:
            failed.append(item)
    return successful, failed

def save_results(successful, failed):
    with open('successful_downloads.json', 'w', encoding='utf-8') as f:
        json.dump(successful, f, indent=2, ensure_ascii=False)
    with open('failed_downloads.json', 'w', encoding='utf-8') as f:
        json.dump(failed, f, indent=2, ensure_ascii=False)

def main():
    try:
        with open('minecraft_schematics_data.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print("Файл minecraft_schematics_data.json не найден.")
        return

    total = len(data)
    print(f"Найдено {total} ссылок для скачивания.\n")

    log_filename = LOG_DIR / f"download_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    log_file = str(log_filename)
    log_message(f"Начало сессии, всего ссылок: {total}", log_file)

    page = create_page(BASE_DOWNLOAD_DIR)

    started_count = 0
    processed_items = []

    try:
        with tqdm(total=total, desc="Инициирование загрузок", unit="файл", dynamic_ncols=True) as pbar:
            for idx, item in enumerate(data, 1):
                url = item.get('download_url')
                filename = item.get('filename', 'unknown')

                pbar.set_description(f"Файл {idx}/{total}")
                log_message(f"[{idx}/{total}] Обработка: {filename}", log_file)

                if not url:
                    log_message("  Нет ссылки — пропущено", log_file)
                    processed_items.append(item)
                    pbar.update(1)
                    continue

                folder_name = sanitize_folder_name(filename)
                download_dir = BASE_DOWNLOAD_DIR / folder_name
                download_dir.mkdir(exist_ok=True)

                success_start = False
                for attempt in range(1, RETRIES + 1):
                    if attempt > 1:
                        log_message(f"  Повторная попытка {attempt}", log_file)
                    if initiate_download(page, url, download_dir, log_file):
                        started_count += 1
                        success_start = True
                        break
                    # Не делаем паузу при неудаче, сразу пробуем ещё раз

                if not success_start:
                    log_message("  ❌ Не удалось начать загрузку", log_file)

                processed_items.append(item)
                pbar.set_postfix(начато=started_count, всего=total)
                pbar.update(1)
                # Убрали time.sleep(0.3), чтобы не замедлять
    finally:
        page.quit()

    log_message(f"Инициирование завершено. Начато загрузок: {started_count}/{total}", log_file)
    print(f"\nИнициирование завершено. Ожидайте завершения всех загрузок...")

    # Финальная проверка
    print("Проверка результатов...")
    successful, failed = finalize_results(processed_items)
    save_results(successful, failed)

    log_message(f"Финальная проверка. Успешно завершённых: {len(successful)}/{total}", log_file)
    print(f"Успешно завершённых загрузок: {len(successful)}")
    print(f"Неудачных: {len(failed)}")
    print(f"Результаты сохранены в successful_downloads.json и failed_downloads.json")
    print(f"Лог сохранён в {log_file}")

if __name__ == "__main__":
    main()

In [48]:
import json


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

In [ ]:
from pathlib import Path
import os
from tqdm import tqdm
import shutil

data_dir = Path("./downloads")
new_data_dir = Path("./filtered_downloads")
new_data_dir.mkdir()

successed = read_json("successful_downloads.json")
successed_filenames = [f["filename"] for f in successed]

for folder in tqdm(os.listdir(data_dir)):
    file = os.listdir(data_dir / folder)[0]


100%|██████████| 4336/4336 [00:05<00:00, 745.34it/s]


In [47]:
final_schematics = []

for file in os.listdir(new_data_dir):
    idx = successed_filenames.index(file)
    final_schematics.append(successed[idx])

with open("final_schem_data.json", "w") as f:
    json.dump(final_schematics, f, indent=4)

Перемещаем в свои подпапки

In [ ]:
import os
from collections import Counter

data_dir = Path("../../data/web_data/")

os.makedirs("../../data/web_data/abfielder_litematic", exist_ok=True)
os.makedirs("../../data/web_data/abfielder_schematic", exist_ok=True)
os.makedirs("../../data/web_data/abfielder_schem", exist_ok=True)

for file in tqdm(os.listdir(data_dir / "abfielder_downloads")):
    file = Path(file)
    path = data_dir / "abfielder_downloads" / file 
    os.rename(path, data_dir / f"abfielder_{file.suffix[1:]}" / file)
    


0it [00:00, ?it/s]


In [119]:
data = read_json("../../data/web_data/final_schem_data.json")

for i in range(len(data)):
    data[i]["id"] = i


with open("../../data/web_data/new_final_schem_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

In [129]:
data = read_json("../../data/web_data/final_schem_data.json")

for i in tqdm(range(len(data))):
    the_id = str(data[i]["id"])
    filename = data[i]["filename"]
    folder = Path(f"../../data/web_data/abfielder_{Path(filename).suffix[1:]}")
    files = os.listdir(folder)
    os.rename(folder / filename, folder / (the_id + Path(filename).suffix))

  0%|          | 0/4336 [00:00<?, ?it/s]

Перекладывает лайтматики каждый в свою подпапку

In [131]:
import shutil


data_dir = Path("../../data/web_data/")
os.makedirs(data_dir / "abfielder_litematic_dirs", exist_ok=True)

for file in tqdm(os.listdir(data_dir / "abfielder_litematic")):
    os.makedirs(data_dir / "abfielder_litematic_dirs" / file)
    shutil.copy(
        data_dir / "abfielder_litematic" / file,
        data_dir / "abfielder_litematic_dirs" / file / file
    )

  0%|          | 0/4147 [00:00<?, ?it/s]

Конвертим в схематики

In [132]:
import os
import subprocess
from pathlib import Path
from tqdm.notebook import tqdm  # нужно установить: pip install tqdm

def find_litematic_file(folder: Path):
    """Возвращает путь к первому файлу с расширением .litematic в папке."""
    litematic_files = list(folder.glob("*.litematic"))
    if not litematic_files:
        return None
    if len(litematic_files) > 1:
        tqdm.write(f"Предупреждение: в {folder} найдено несколько .litematic файлов. Берём первый.")
    return litematic_files[0]

def process_subfolders(root_dir: Path, jar_path: Path):
    """
    Обходит все подпапки внутри root_dir.
    Для каждой подпапки, содержащей .litematic файл, запускает конвертацию.
    """
    # Ищем все подпапки первого уровня
    subfolders = [f for f in root_dir.iterdir() if f.is_dir()]
    
    # Счётчики для итоговой статистики
    success_count = 0
    error_count = 0
    skip_count = 0

    # Оборачиваем список в tqdm для отображения прогресса
    for sub in tqdm(subfolders, desc="Обработка папок", unit="папка"):
        litematic = find_litematic_file(sub)
        if litematic is None:
            tqdm.write(f"Пропускаем {sub.name}: нет .litematic файла.")
            skip_count += 1
            continue

        cmd = ["java", "-jar", str(jar_path), "--convert", litematic.name]
        # tqdm.write(f"Выполняется в {sub.name}: {' '.join(cmd)}")
        
        try:
            result = subprocess.run(cmd, cwd=str(sub), capture_output=True, text=True)
            if result.returncode != 0:
                tqdm.write(f"Ошибка при обработке {litematic.name}:")
                tqdm.write(result.stderr)
                error_count += 1
            else:
                # tqdm.write(f"Успешно обработан {litematic.name}")
                success_count += 1
        except Exception as e:
            tqdm.write(f"Исключение при запуске команды для {litematic.name}: {e}")
            error_count += 1

    # Итоговая статистика
    tqdm.write("\n=== Статистика ===")
    tqdm.write(f"Успешно: {success_count}")
    tqdm.write(f"Ошибок: {error_count}")
    tqdm.write(f"Пропущено (нет .litematic): {skip_count}")
    tqdm.write(f"Всего папок: {len(subfolders)}")

if __name__ == "__main__":
    root_directory = Path("../../data/web_data/abfielder_litematic_dirs")
    jar_file = Path("../../data/Lite2Edit.jar").resolve()

    if not jar_file.exists():
        print(f"Ошибка: файл {jar_file} не найден.")
        exit(1)

    process_subfolders(root_directory, jar_file)

Обработка папок:   0%|          | 0/4148 [00:00<?, ?папка/s]

Пропускаем 4335: нет .litematic файла.

=== Статистика ===
Успешно: 4147
Ошибок: 0
Пропущено (нет .litematic): 1
Всего папок: 4148


In [133]:
len(read_json("../../data/web_data/final_schem_data.json"))

4336

Удаляем чепуху всякую

In [ ]:
lite_dir = Path("../../data/web_data/abfielder_litematic_dirs")

for dir in os.listdir(lite_dir):
    files = os.listdir(lite_dir / dir)
    if len(files) != 2:
        print(dir)
        shutil.rmtree(lite_dir / dir)

In [143]:
len(read_json("../../data/web_data/final_schem_data.json"))

4297

Перемещаем лайтматики

In [ ]:
os.makedirs("../../data/web_data/abfielder_schem_all", exist_ok=True)
dst = Path("../../data/web_data/abfielder_schem_all")

for dir in os.listdir(lite_dir):
    dir = Path(dir)
    schem_file = lite_dir / dir / (dir.stem + ".schem")
    shutil.copy2(schem_file, dst / schem_file.name)

Перемешаем схематики

In [147]:
schem_dir = Path("../../data/web_data/abfielder_schem")

for file in os.listdir(schem_dir):
    shutil.copy2(schem_dir / file, dst / file)